# 09 — Video Understanding

## What
Video understanding extends image models to the temporal dimension. Key tasks: action recognition, temporal grounding (find when an event occurs), video captioning, and video question answering. The core challenge is efficient temporal modelling — video has ~24-60x more data than single images.

## Why
Video is the dominant modality on the internet. YouTube, TikTok, and enterprise surveillance generate petabytes daily. Autonomous driving, sports analytics, manufacturing quality control, and medical endoscopy all require temporal understanding that images alone cannot provide.

## Community context
SlowFast networks (Feichtenhofer et al., 2019) used two-pathway CNNs for different temporal resolutions. TimeSformer and Video Swin Transformer applied attention to video. VideoMAE used masked autoencoders with 90% masking to learn temporal representations. Gemini 1.5's 1M context window enabled long-video understanding. InternVideo2 achieves state-of-art on multiple benchmarks.

In [ ]:
# Video temporal modelling: sampling, pooling, temporal attention
import numpy as np

def softmax(x, axis=-1):
    e = np.exp(x - x.max(axis=axis, keepdims=True))
    return e / e.sum(axis=axis, keepdims=True)

class VideoFrameSampler:
    """Temporal frame sampling strategies"""

    @staticmethod
    def uniform_sample(n_frames_total, n_frames_sample):
        return np.linspace(0, n_frames_total-1, n_frames_sample, dtype=int)

    @staticmethod
    def random_sample(n_frames_total, n_frames_sample, seed=None):
        rng = np.random.default_rng(seed)
        return np.sort(rng.choice(n_frames_total, n_frames_sample, replace=False))

    @staticmethod
    def dense_sample(n_frames_total, n_frames_sample, clip_len=16):
        """Sample n_frames_sample clips of clip_len frames"""
        starts = np.linspace(0, n_frames_total-clip_len, n_frames_sample, dtype=int)
        return [(s, s+clip_len) for s in starts]

class TemporalAttentionPooling:
    """
    Aggregate per-frame features into a single video representation
    using learned temporal attention weights.
    """
    def __init__(self, feat_dim):
        self.W = np.random.randn(feat_dim, 1) * 0.01

    def __call__(self, frame_features):
        """frame_features: (T, D) -> video_repr: (D,)"""
        scores = frame_features @ self.W   # (T, 1)
        weights = softmax(scores.squeeze())  # (T,)
        return weights @ frame_features     # (D,), weighted sum

class TemporalTransformerBlock:
    """Attention over time dimension of video frames"""
    def __init__(self, feat_dim, n_heads=4):
        self.n_heads = n_heads
        self.head_dim = feat_dim // n_heads
        scale = np.sqrt(feat_dim)
        self.Wq = np.random.randn(feat_dim, feat_dim) / scale
        self.Wk = np.random.randn(feat_dim, feat_dim) / scale
        self.Wv = np.random.randn(feat_dim, feat_dim) / scale
        self.Wo = np.random.randn(feat_dim, feat_dim) / scale

    def __call__(self, frame_features):
        """frame_features: (T, D) -> (T, D) with temporal context"""
        Q = frame_features @ self.Wq
        K = frame_features @ self.Wk
        V = frame_features @ self.Wv
        attn = softmax(Q @ K.T / np.sqrt(self.head_dim))
        return frame_features + attn @ V @ self.Wo

np.random.seed(42)
T_total = 300   # 10 second video at 30fps
T_sample = 16   # sample 16 frames
feat_dim = 128  # per-frame feature dimension

sampler = VideoFrameSampler()
indices = sampler.uniform_sample(T_total, T_sample)

# Simulate per-frame features from a frozen ViT
frame_feats = np.random.randn(T_sample, feat_dim)

# Temporal attention
temporal_block = TemporalTransformerBlock(feat_dim)
temporal_feats = temporal_block(frame_feats)

# Pool to video-level representation
pool = TemporalAttentionPooling(feat_dim)
video_repr = pool(temporal_feats)

print('Video Understanding Pipeline:')
print(f'  Total frames:       {T_total} ({T_total/30:.1f}s at 30fps)')
print(f'  Sampled frames:     {T_sample}')
print(f'  Frame features:     {frame_feats.shape}')
print(f'  Temporal attention: {temporal_feats.shape}')
print(f'  Video repr:         {video_repr.shape}  (aggregated video embedding)')
print(f'\nSampled frame indices: {indices}')
print('This vector can be used for action classification or fed to a VLM')

## Action Recognition Evaluation Metrics

Top-1 and Top-5 accuracy on Kinetics-400/700, Something-Something-v2, and ActivityNet are the standard benchmarks. Mean Average Precision (mAP) is used for temporal action detection where actions have start/end times.

In [ ]:
# Action recognition evaluation
def topk_accuracy(scores, labels, k=5):
    """Top-k accuracy across a batch"""
    topk_preds = np.argsort(scores, axis=-1)[:, -k:]
    correct = sum(labels[i] in topk_preds[i] for i in range(len(labels)))
    return correct / len(labels)

def temporal_iou(pred_start, pred_end, gt_start, gt_end):
    intersection = max(0, min(pred_end, gt_end) - max(pred_start, gt_start))
    union = max(pred_end, gt_end) - min(pred_start, gt_start)
    return intersection / (union + 1e-8)

np.random.seed(7)
N_classes = 400  # Kinetics-400
N_samples = 200
labels = np.random.randint(0, N_classes, N_samples)

# Simulate model scores: model is biased toward correct class
scores = np.random.randn(N_samples, N_classes)
for i, l in enumerate(labels):
    scores[i, l] += 3.0  # boost true class

top1 = topk_accuracy(scores, labels, k=1)
top5 = topk_accuracy(scores, labels, k=5)
print(f'Action Recognition (simulated Kinetics-400 eval):')
print(f'  Top-1 accuracy: {top1:.3f}')
print(f'  Top-5 accuracy: {top5:.3f}')

# Temporal detection IoU
print(f'\nTemporal IoU examples:')
for pred, gt in [((10,25),(12,27)),(( 0,10),(15,25)),((5,20),(8,22))]:
    iou = temporal_iou(*pred, *gt)
    print(f'  pred={pred} gt={gt} IoU={iou:.3f}')